### Importando bibliotecas necessárias

In [28]:
import pandas as pd

### Leitura do dataset

In [29]:
df_trancricoes = pd.read_csv('../../data/call_recordings.csv')
df_trancricoes.head()

,id,Type,Sentiment,Name,Order Number,Product Number,Transcript
0,call_recording_01,Product Inquiry,Neutral,Sarah Miller,NaN,AC-7892,"Hello, I'm Sarah Miller. I'm calling to inquir..."
1,call_recording_02,Complaint,Angry,John Davis,123456.0,FR-4401,I am extremely dissatisfied with my recent ord...
2,call_recording_03,Technical Issue,Frustrated,Maria Rodriguez,987654.0,LAP-2110,"Hi, this is Maria Rodriguez. I'm having troubl..."
3,call_recording_04,Compliment,Happy,Robert Smith,246801.0,DW-6543,I just wanted to call and say how pleased I am...
4,call_recording_05,Order Placement,Neutral,Jessica Brown,NaN,OV-1357 & MW-8642,"Hello, my name is Jessica Brown. I'd like to p..."


### Validar linhas e colunas do dataset

In [30]:
df_trancricoes.isnull().sum()

id                0
Type              0
Sentiment         0
Name              0
Order Number      8
Product Number    0
Transcript        0
dtype: int64

### Normalizar a coluna transcription

In [31]:
df_trancricoes.duplicated().sum()

np.int64(0)

In [32]:
df_trancricoes.count()

id                20
Type              20
Sentiment         20
Name              20
Order Number      12
Product Number    20
Transcript        20
dtype: int64

In [33]:
df_trancricoes = df_trancricoes[df_trancricoes['Transcript'].str.len() > 15 ]
df_trancricoes.count()

id                20
Type              20
Sentiment         20
Name              20
Order Number      12
Product Number    20
Transcript        20
dtype: int64

In [34]:
### Padronizando para lowercase
df_trancricoes['Transcript'] = df_trancricoes['Transcript'].str.lower()
df_trancricoes['Transcript'] = df_trancricoes['Transcript'].str.strip()
# Remover placeholders de ruído e artefatos de transcrição
df_trancricoes['Transcript'] = df_trancricoes['Transcript'].str.replace(
    r'\[inaudível\]|\[inaudivel\]|\[ruído\]|\[ruido\]|\[.*?\]',
    '',
    regex=True
)
df_trancricoes['Transcript'] = df_trancricoes['Transcript'].str.replace(r'\d{1,2}:\d{2}:\d{2}', '', regex=True)
df_trancricoes.head()

,id,Type,Sentiment,Name,Order Number,Product Number,Transcript
0,call_recording_01,Product Inquiry,Neutral,Sarah Miller,NaN,AC-7892,"hello, i'm sarah miller. i'm calling to inquir..."
1,call_recording_02,Complaint,Angry,John Davis,123456.0,FR-4401,i am extremely dissatisfied with my recent ord...
2,call_recording_03,Technical Issue,Frustrated,Maria Rodriguez,987654.0,LAP-2110,"hi, this is maria rodriguez. i'm having troubl..."
3,call_recording_04,Compliment,Happy,Robert Smith,246801.0,DW-6543,i just wanted to call and say how pleased i am...
4,call_recording_05,Order Placement,Neutral,Jessica Brown,NaN,OV-1357 & MW-8642,"hello, my name is jessica brown. i'd like to p..."


### Converter a coluna 'transcription' para vetor de características usando TF-IDF

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
X = vectorizer.fit_transform(df_trancricoes['Transcript'])

In [37]:
df_vetorizado = df_trancricoes[['Transcript']].copy()
df_vetorizado['tfidf_vector'] = [row.toarray().ravel() for row in X]
df_vetorizado.head()

,Transcript,tfidf_vector
0,"hello, i'm sarah miller. i'm calling to inquir...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,i am extremely dissatisfied with my recent ord...,"[0.0, 0.0, 0.1816734892961844, 0.0, 0.0, 0.0, ..."
2,"hi, this is maria rodriguez. i'm having troubl...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.1871218536643018, ..."
3,i just wanted to call and say how pleased i am...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.192..."
4,"hello, my name is jessica brown. i'd like to p...","[0.0, 0.0, 0.0, 0.21507955799280892, 0.0, 0.0,..."


### Aplicar o modelo de agrupamento K-means para classificar as transcrições

In [47]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

kmeans = KMeans(n_clusters=12, random_state=42, n_init='auto')
clusters = kmeans.fit_predict(X)

df_trancricoes['cluster'] = clusters
print('Silhouette:', silhouette_score(X, clusters))
df_trancricoes[['Transcript', 'cluster']].head(20)

Silhouette: 0.020290711406548617


,Transcript,cluster
0,"hello, i'm sarah miller. i'm calling to inquir...",3
1,i am extremely dissatisfied with my recent ord...,2
2,"hi, this is maria rodriguez. i'm having troubl...",9
3,i just wanted to call and say how pleased i am...,1
4,"hello, my name is jessica brown. i'd like to p...",6
5,"hi, this is david wilson. i'm looking at the s...",8
6,i'm calling about the hd-2771 hard drive i ord...,4
7,"hello, my name is michael brown. i need some t...",0
8,"hi, this is amanda white. i'd like to place an...",5
9,i'm calling to express my satisfaction with th...,1
